# Telco Customer Churn — EDA & Interview Insights

This notebook documents exploratory findings used to shape feature engineering and model choices.

**Goal:** identify which customers are likely to leave so retention spend can be prioritized.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data import load_modeling_frame

df = load_modeling_frame()
print(df.shape)
df.head()

## 1. Target balance

Churn is the minority class (~26%). Accuracy alone is misleading — we track ROC AUC, recall, and a business cost threshold.

In [ ]:
churn_rate = df["Churn"].mean()
print(f"Churn rate: {churn_rate:.1%}")
df["Churn"].value_counts().rename({0: "Stay", 1: "Churn"}).plot(
    kind="bar", color=["#2a6f97", "#c1121f"], title="Class balance"
)
plt.ylabel("Customers")
plt.tight_layout()
plt.show()

## 2. Contract type vs churn

**Interview point:** Month-to-month customers churn much more — low switching cost. Longer contracts are a strong retention lever.

In [ ]:
contract_churn = df.groupby("Contract")["Churn"].mean().sort_values(ascending=False)
print(contract_churn)
contract_churn.plot(kind="bar", color="#2a6f97", title="Churn rate by contract")
plt.ylabel("Churn rate")
plt.tight_layout()
plt.show()

## 3. Tenure risk

**Interview point:** Customers in the first year are the riskiest. Onboarding and early-life support matter more than late-stage discounts alone.

In [ ]:
tenure_churn = df.groupby("tenure_bucket")["Churn"].mean().reindex(
    ["0-12", "13-24", "25-48", "49+"]
)
print(tenure_churn)
tenure_churn.plot(kind="bar", color="#2a6f97", title="Churn rate by tenure bucket")
plt.ylabel("Churn rate")
plt.tight_layout()
plt.show()

## 4. Charges and internet service

**Interview point:** Higher monthly charges + fiber often correlate with churn — may signal price/value mismatch rather than "high spenders are loyal."

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df.boxplot(column="MonthlyCharges", by="Churn", ax=axes[0])
axes[0].set_title("Monthly charges by churn")
axes[0].get_figure().suptitle("")

internet_churn = df.groupby("InternetService")["Churn"].mean().sort_values(ascending=False)
internet_churn.plot(kind="bar", ax=axes[1], color="#2a6f97", title="Churn by internet service")
axes[1].set_ylabel("Churn rate")
plt.tight_layout()
plt.show()
print(internet_churn)

## 5. Payment method & add-on depth

**Interview points:**
- Electronic check payers churn more than automatic payment methods.
- Higher `service_count` (security, backup, support, streaming) tends to mean lower churn — product depth creates stickiness.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
pay = df.groupby("PaymentMethod")["Churn"].mean().sort_values(ascending=False)
pay.plot(kind="barh", ax=axes[0], color="#2a6f97", title="Churn by payment method")
axes[0].set_xlabel("Churn rate")

svc = df.groupby("service_count")["Churn"].mean()
svc.plot(kind="bar", ax=axes[1], color="#2a6f97", title="Churn by service_count")
axes[1].set_ylabel("Churn rate")
plt.tight_layout()
plt.show()

## Interview summary (memorize these)

1. **Problem:** binary classification with ~26% positive class — optimize ranking (AUC) + business threshold, not raw accuracy.
2. **Strong signals:** contract type, tenure, internet service, payment method, monthly charges.
3. **Features added:** `tenure_bucket`, `avg_monthly_spend`, `service_count` — simple and explainable.
4. **Imbalance:** stratified split + `class_weight='balanced'` before reaching for SMOTE.
5. **Models compared:** Logistic Regression, Random Forest, Gradient Boosting — pick by ROC AUC.
6. **Threshold:** tuned so missing a churner costs more than a wasted retention offer.
7. **Limitation:** observational telco snapshot — no causal claims; drift possible if pricing/products change.